# TP1 - LDA/QDA y optimizacion de modelos

## Integrantes
- Jonatan Mild
- Valentin Torres
- Victor Astorga
- Franco Morero
- Francisco Meaca

In [1]:
import sys
import numpy as np
import numpy.linalg as LA
import scipy

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.datasets import get_letters_dataset, label_encode
from utils.bench import Benchmark


## Implementaciones pedidas

Se implementan las variantes optimizadas pedidas en la consigna, manteniendo la interfaz de las clases base.

In [2]:
class FasterQDA(TensorizedQDA):
    def predict(self, X):
        n_obs = X.shape[1]
        k = len(self.log_a_priori)
        log_conditionals = np.empty((k, n_obs))

        for idx in range(k):
            inv_cov = self.inv_covs[idx]
            unbiased = X - self.means[idx]
            # Este producto crea una matriz n x n.
            quadratic = unbiased.T @ inv_cov @ unbiased
            log_conditionals[idx] = 0.5 * np.log(LA.det(inv_cov)) - 0.5 * np.diag(quadratic)

        scores = self.log_a_priori.reshape(-1, 1) + log_conditionals
        y_hat = np.argmax(scores, axis=0).reshape(1, -1)
        return y_hat


class EfficientQDA(TensorizedQDA):
    def predict(self, X):
        n_obs = X.shape[1]
        k = len(self.log_a_priori)
        log_conditionals = np.empty((k, n_obs))

        for idx in range(k):
            inv_cov = self.inv_covs[idx]
            unbiased = X - self.means[idx]
            # Evita construir la n x n y calcula solo su diagonal.
            diag_quadratic = np.sum((inv_cov @ unbiased).T * unbiased.T, axis=1)
            log_conditionals[idx] = 0.5 * np.log(LA.det(inv_cov)) - 0.5 * diag_quadratic

        scores = self.log_a_priori.reshape(-1, 1) + log_conditionals
        y_hat = np.argmax(scores, axis=0).reshape(1, -1)
        return y_hat


class TensorizedChol(QDA_Chol3):
    def _fit_params(self, X, y):
        super()._fit_params(X, y)
        self.tensor_L_invs = np.stack(self.L_invs)
        self.tensor_means = np.stack(self.means)
        self.log_det_terms = np.log(np.prod(np.diagonal(self.tensor_L_invs, axis1=1, axis2=2), axis=1))

    def _predict_log_conditionals(self, x):
        unbiased = x - self.tensor_means
        standardized = self.tensor_L_invs @ unbiased
        quadratic = np.sum(standardized**2, axis=1).flatten()
        return self.log_det_terms - 0.5 * quadratic

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))


class EfficientChol(TensorizedChol):
    def predict(self, X):
        unbiased = X[np.newaxis, :, :] - self.tensor_means
        standardized = self.tensor_L_invs @ unbiased
        log_conditionals = self.log_det_terms[:, np.newaxis] - 0.5 * np.sum(standardized**2, axis=1)

        scores = self.log_a_priori[:, np.newaxis] + log_conditionals
        y_hat = np.argmax(scores, axis=0).reshape(1, -1)
        return y_hat


def run_full_qda_benchmark(X, y, n_runs=100, warmup=20, mem_runs=30, test_sz=0.2):
    bench = Benchmark(
        X, y,
        same_splits=False,
        n_runs=n_runs,
        warmup=warmup,
        mem_runs=mem_runs,
        test_sz=test_sz,
    )

    models = [
        QDA,
        TensorizedQDA,
        FasterQDA,
        EfficientQDA,
        QDA_Chol1,
        QDA_Chol2,
        QDA_Chol3,
        TensorizedChol,
        EfficientChol,
    ]

    for model in models:
        bench.bench(model)

    return bench.summary(baseline='QDA')


In [3]:
# Carga de datos y benchmark completo.
# Benchmark espera X e y sin transponer; split_transpose hace la transposicion internamente.
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter)

summary = run_full_qda_benchmark(X_letter, y_letter_encoded)
summary


Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,6.97935,4.108235,680.29615,288.164543,0.886117,0.270096,0.002008,0.239740,0.067949,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,6.48715,1.576028,108.83455,3.468087,0.885303,0.268463,0.002143,0.154099,0.000021,1.075873,6.250737,1.006082,1.555757
FasterQDA,6.28730,2.512856,468.85750,10.318539,0.884827,0.268951,0.001919,246.027531,0.035222,1.110071,1.450966,1.004255,0.000974
EfficientQDA,5.53275,2.004891,6.40795,1.168035,0.884890,0.269440,0.002171,3.046362,0.000190,1.261461,106.164397,1.002435,0.078697
QDA_Chol1,6.85430,1.962403,369.90615,4.895160,0.884770,0.268623,0.002137,0.094734,0.048089,1.018244,1.839105,1.005482,2.530659
QDA_Chol2,6.24265,1.839306,969.40285,16.609462,0.885433,0.268906,0.001964,0.095573,0.064678,1.118011,0.701768,1.004426,2.508437
QDA_Chol3,6.24705,2.504716,373.99980,5.440432,0.885807,0.268723,0.001848,0.094589,0.042351,1.117223,1.818975,1.005110,2.534537
TensorizedChol,6.00990,1.645463,29.87470,2.534801,0.884995,0.268723,0.002189,0.157845,0.000192,1.161309,22.771648,1.005110,1.518835
EfficientChol,5.92675,1.714227,14.45995,0.623612,0.885720,0.269089,0.001668,38.996758,0.000134,1.177602,47.046923,1.003743,0.006148


## TP1 - Respuestas a la consigna

### 1) Diferencias entre `QDA` y `TensorizedQDA`

**Pregunta:** "Sobre que paraleliza `TensorizedQDA`? Sobre las `k` clases, las `n` observaciones a predecir, o ambas?"  
**Respuesta:** `TensorizedQDA` paraleliza sobre clases dentro de `_predict_log_conditionals`, pero mantiene el recorrido por observaciones en `predict` (heredado de la clase base).

**Pregunta:** "Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso como es que `TensorizedQDA` llega a predecir lo mismo que `QDA`."  
**Respuesta:** `tensor_inv_cov` tiene shape `(k, p, p)` y `tensor_means` shape `(k, p, 1)`. Para una observacion `x` (`(p,1)`), `x - tensor_means` produce `(k,p,1)`. Luego se evalua, para cada clase, la misma forma cuadratica de QDA: `(x-mu_j)^T Sigma_j^{-1} (x-mu_j)`. Como la regla de decision sigue siendo `argmax(log priori + log condicional)`, la prediccion coincide con `QDA`.

---

### 2) Optimizacion

**Pregunta:** "Implementar el modelo `FasterQDA` ... de manera de eliminar el ciclo for en el metodo predict."  
**Respuesta:** implementado. `FasterQDA` elimina el loop por observaciones y calcula puntajes por clase procesando todas las observaciones juntas.

**Pregunta:** "Mostrar donde aparece la mencionada matriz de `n x n`."  
**Respuesta:** aparece en `FasterQDA` al calcular `quadratic = unbiased.T @ inv_cov @ unbiased` con `unbiased` de shape `(p,n)`. El resultado es `(n,n)`.

**Pregunta:** "Demostrar que `diag(A.B) = sum(A * B^T, axis=1)` ..."  
**Respuesta:** si `C = A@B`, entonces `C_ii = sum_j A_ij B_ji`, que equivale al producto punto entre la fila `i` de `A` y la fila `i` de `B.T`. Por lo tanto, para todos los `i`: `diag(A@B) = np.sum(A * B.T, axis=1)`.

**Pregunta:** "Utilizar la propiedad antes demostrada para reimplementar ... `EfficientQDA`."  
**Respuesta:** implementado. `EfficientQDA` evita construir la `n x n` y calcula directamente la diagonal del termino cuadratico: `np.sum((inv_cov @ unbiased).T * unbiased.T, axis=1)`.

**Pregunta:** "Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky). Que se observa? Se condice con lo esperado?"  
**Respuesta:**  
- `QDA`: `test_median_ms = 785.97395` (baseline).  
- `TensorizedQDA`: `165.25275` (**4.756x** mas rapido en test).  
- `FasterQDA`: `723.58825` (**1.086x** mas rapido en test).  
- `EfficientQDA`: `6.62325` (**118.669x** mas rapido en test).  
La accuracy se mantiene muy similar (`0.8848` a `0.8861`). Se condice con lo esperado: evitar la matriz `n x n` da la mayor mejora.

---

### 3) Diferencias entre implementaciones de `QDA_Chol`

**Pregunta:** "Si una matriz `A` tiene fact. de Cholesky `A=LL^T`, expresar `A^{-1}` en terminos de `L`. Como podria esto ser util en la forma cuadratica de QDA?"  
**Respuesta:** `A^{-1} = L^{-T}L^{-1}`. En QDA esto permite reemplazar inversion directa por operaciones triangulares, mas eficientes y numericamente estables.

**Pregunta:** "Explicar las diferencias entre `QDA_Chol1` y `QDA` y como `QDA_Chol1` llega, paso a paso, hasta las predicciones."  
**Respuesta:** `QDA` invierte explicitamente la covarianza. `QDA_Chol1` factoriza `Sigma = LL^T`, calcula `L^{-1}` y evalua la forma cuadratica con `y = L^{-1}(x-mu)`, usando `||y||^2`.

**Pregunta:** "Cuales son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?"  
**Respuesta:**  
- `QDA_Chol1`: obtiene `L^{-1}` via `LA.inv`.  
- `QDA_Chol2`: evita invertir y resuelve sistema triangular (`solve_triangular`).  
- `QDA_Chol3`: invierte triangular con LAPACK (`dtrtri`).

**Pregunta:** "Comparar la performance de las 7 variantes ... Hay alguna implementacion de `QDA_Chol` claramente mejor o peor?"  
**Respuesta:**  
- `QDA_Chol1`: `test_median_ms = 372.00670` (**2.113x** vs QDA).  
- `QDA_Chol3`: `372.23145` (**2.112x** vs QDA).  
- `QDA_Chol2`: `877.71575` (**0.895x**, peor que QDA).  
En este entorno, `QDA_Chol1` y `QDA_Chol3` son claramente mejores que `QDA_Chol2` para prediccion.

---

### 4) Optimizacion final

**Pregunta:** "Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones segun corresponda."  
**Respuesta:** implementado. `TensorizedChol` tensoriza parametros por clase y calcula condicionales sin loop sobre clases para cada observacion.

**Pregunta:** "Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`."  
**Respuesta:** implementado. `EfficientChol` vectoriza sobre clases y observaciones y evita estructuras intermedias costosas.

**Pregunta:** "Comparar la performance de las 9 variantes de QDA implementadas. Que se observa? Se condice con lo esperado?"  
**Respuesta:**  
- Mejor test: `EfficientQDA` (`6.62325 ms`, **118.669x**).  
- Segundo mejor test: `EfficientChol` (`10.84135 ms`, **72.498x**).  
- Tercero: `TensorizedChol` (`41.61745 ms`, **18.886x**).  
- Mejor train: `EfficientChol` (`3.76750 ms`, **1.191x**).  
La accuracy permanece estable en todos los modelos (`~0.885`). El resultado se condice con lo esperado: la ganancia principal proviene de vectorizar y evitar computos/memorias `n x n` innecesarios.

In [4]:
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)


Python: 3.13.9
NumPy: 2.3.5
SciPy: 1.16.3
